# Assignment 27: Chatbot  with Conversation History Using Langchain
---

# Part 1: Message and Message Placeholders


## TASK 1: Understadning Chat Message

1. What are SystemMessage, HumanMessage, And AIMessage
- `SystemMessage`:Defines the behavior, role, and instructions for the LLM. 
- `HumanMessage`:Represents the user's input/question.                     
- `AIMessage`:Represents the LLM's previous response.   

2. Why message-based prompting is better than single prompts
- Message-based prompting separates system instructions, user input, and previous AI responses. 
- This makes conversation history easier to manage and allows the LLM to understand the context of a multi-turn conversation.

## TASK 2: MessagePlaceholder Usage

In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful customer support chatbot. "
        "Use the conversation history to answer accurately."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

In [4]:
MessagesPlaceholder(variable_name="chat_history")

MessagesPlaceholder(variable_name='chat_history')

In [5]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

In [6]:
from langchain_core.messages import HumanMessage, AIMessage

In [7]:


chat_history = [
    HumanMessage(content="My name is Arun."),
    AIMessage(content="Hello Arun! How can I help you?"),
    HumanMessage(content="I have an issue with my order."),
    AIMessage(content="Sure. Please provide your order ID.")
]

In [ ]:
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "input": "My order number is 12345."
})

response = llm.invoke(formatted_prompt)



In [9]:
print(response.content)

Thank you for providing your order number. How can I assist you with your order, Arun?


# PART 2 — Conversation History Management
## Task 3: Basic Message History

In [12]:
chat_history = []

def chat(user_input):
    global chat_history

    formatted_prompt = prompt.invoke({
        "chat_history": chat_history,
        "input": user_input
    })

    response = llm.invoke(formatted_prompt)
    chat_history.append(
        HumanMessage(content=user_input)
    )
    chat_history.append(
        AIMessage(content=response.content)
    )

    return response.content

In [13]:
chat("My name is Arun")

'Nice to meet you, Arun! How can I assist you today?'

In [14]:
chat("I have a problem with my order.")

"I'm sorry to hear that, Arun. Please provide me with your order details so I can assist you further."

In [15]:
chat("What is my name?")

'Your name is Arun. How can I assist you with your order issue, Arun?'

In [16]:
for message in chat_history:
    print(type(message).__name__, ":", message.content)

HumanMessage : My name is Arun
AIMessage : Nice to meet you, Arun! How can I assist you today?
HumanMessage : I have a problem with my order.
AIMessage : I'm sorry to hear that, Arun. Please provide me with your order details so I can assist you further.
HumanMessage : What is my name?
AIMessage : Your name is Arun. How can I assist you with your order issue, Arun?


## Task 4: Trimming Chat History

In [17]:
MAX_MESSAGES = 6

def trim_history(history):
    if len(history) > MAX_MESSAGES:
        return history[-MAX_MESSAGES:]
    
    return history

In [18]:
chat_history = []

def chat(user_input):
    global chat_history
    chat_history = trim_history(chat_history)

    formatted_prompt = prompt.invoke({
        "chat_history": chat_history,
        "input": user_input
    })

    response = llm.invoke(formatted_prompt)
    chat_history.append(
        HumanMessage(content=user_input)
    )
    chat_history.append(
        AIMessage(content=response.content)
    )
    chat_history = trim_history(chat_history)

    return response.content

In [19]:
chat("My name is Arun.")

'Nice to meet you, Arun! How can I assist you today?'

In [21]:
chat("I ordered a laptop.")

'Thank you for letting me know, Arun. Do you have the order number or tracking information for your laptop delivery? This will help me investigate the delay further.'

In [20]:
chat("The delivery is delayed.")

"I'm sorry to hear that your delivery is delayed, Arun. Please provide me with your order number so I can look into it for you."

In [22]:
chat("Can you help me with the order?")

'Of course, Arun. I can try to help you with your order. Can you please provide me with any details you have, such as the order number, tracking information, or the name of the online store where you made the purchase?'

In [23]:
len(chat_history)

6

In [ ]:
for message in chat_history:
    print(f"{type(message).__name__}: {message.content}")

# PART 3 — Q&A Chatbot with Message History
## Task 5: Build Q&A Chatbot

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

def ask_question(question):
    global chat_history
    chat_history = trim_history(chat_history)

    prompt_messages = prompt.invoke({
        "chat_history": chat_history,
        "input": question
    })

    response = llm.invoke(prompt_messages)

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response.content))

    chat_history = trim_history(chat_history)

    return response.content

In [ ]:
ask_question("Explain Python lists.")

In [ ]:
ask_question("Give an example.")

In [ ]:
ask_question("What about tuples?")

In [ ]:
ask_question("What is the difference between lists and tuples?")

## Task 6: Build Stateful Chatbot Application

In [ ]:
chat_history = []

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    response = ask_question(user_input)

    print("Chatbot:", response)

## Task 7: Observations & Insights


1. Why is chat history important?
- Chat history allows the chatbot to understand previous messages and answer follow-up questions contextually. Without history, questions such as "Give an example" may be ambiguous.

2. Trade-offs between long memory and performance
- Longer history provides more context but increases token usage, processing time, and cost. Shorter history improves performance and reduces token usage but may lose important context.

3. When to summarize vs trim history
- Trim history is suitable when recent messages contain enough context and older messages are not important.
- Summarization is better when older information is important but the conversation is too long. A summary preserves important information while reducing the number of tokens sent to the LLM.

4. Difference between MessagePlaceholder and memory
- MessagesPlaceholder is a prompt component used to insert a list of conversation messages into a prompt.
- Memory is the mechanism used to store and manage conversation state/history.